# Tech Challenge Fase 3 — Enriquecimento Socioeconômico

## Objetivo

Construir uma base socioeconômica municipal que será utilizada posteriormente para enriquecer a base individual de alunos.

Nesta etapa serão integradas duas fontes externas:

- **População municipal — IBGE / Base dos Dados**
- **RAIS — mercado formal de trabalho**

Para reduzir o risco de *data leakage* temporal:

- alunos de **2023** utilizarão contexto RAIS de **2022**;
- alunos de **2024** utilizarão contexto RAIS de **2023**.

A população será utilizada no próprio ano de referência do aluno.

O resultado será persistido em:

`workspace.alfabetizacao_silver.socioeconomico_municipio`


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "alfabetizacao_bronze"
SILVER_SCHEMA = "alfabetizacao_silver"

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/socioeconomico"
)

POPULACAO_PATH = (
    f"{VOLUME_PATH}/br_ibge_populacao_municipio.csv.gz"
)

RAIS_PATH = (
    f"{VOLUME_PATH}/rais_municipio_2022_2023.csv"
)

OUTPUT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.socioeconomico_municipio"
)

print("Arquivo de população:", POPULACAO_PATH)
print("Arquivo RAIS:", RAIS_PATH)
print("Tabela de destino:", OUTPUT_TABLE)


## 2. Leitura das fontes

Os arquivos são lidos diretamente do Volume do Databricks.

A leitura é feita inicialmente como texto para que a tipagem seja controlada explicitamente durante o tratamento.


In [ ]:
df_pop_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(POPULACAO_PATH)
)

df_rais_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(RAIS_PATH)
)

print("Colunas da população:")
print(df_pop_raw.columns)

print("\nColunas da RAIS:")
print(df_rais_raw.columns)

display(df_pop_raw.limit(5))
display(df_rais_raw.limit(5))


## 3. Tratamento da população

A base de população possui série histórica ampla.

Serão mantidos apenas os anos de **2023 e 2024**, correspondentes ao período dos alunos utilizado no Tech Challenge.


In [ ]:
df_pop = (
    df_pop_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.upper(F.trim(F.col("sigla_uf"))).alias("sigla_uf"),
        F.trim(F.col("id_municipio")).alias("id_municipio"),
        F.col("populacao").cast("long").alias("populacao"),
    )
    .filter(F.col("ano").isin(2023, 2024))
    .filter(F.col("id_municipio").isNotNull())
)

display(
    df_pop
    .orderBy("ano", "id_municipio")
    .limit(10)
)


## 4. Tratamento da RAIS

A RAIS é utilizada com **defasagem de um ano**.

Dessa forma:

- RAIS 2022 → contexto socioeconômico para alunos de 2023;
- RAIS 2023 → contexto socioeconômico para alunos de 2024.

Essa decisão impede o uso de informações posteriores ao período que será previsto.


In [ ]:
df_rais = (
    df_rais_raw
    .select(
        F.col("ano").cast("int").alias("ano_rais"),
        F.trim(F.col("id_municipio")).alias("id_municipio"),
        F.col("quantidade_vinculos_ativos")
            .cast("long")
            .alias("quantidade_vinculos_ativos"),
        F.col("quantidade_vinculos_clt")
            .cast("long")
            .alias("quantidade_vinculos_clt"),
        F.col("quantidade_vinculos_estatutarios")
            .cast("long")
            .alias("quantidade_vinculos_estatutarios"),
    )
    .filter(F.col("ano_rais").isin(2022, 2023))
    .filter(F.col("id_municipio").isNotNull())
    .withColumn(
        "ano",
        F.col("ano_rais") + F.lit(1),
    )
)

display(
    df_rais
    .orderBy("ano_rais", "id_municipio")
    .limit(10)
)


## 5. Cobertura das fontes

Antes da integração, verificamos quantos municípios estão disponíveis em cada fonte por ano.


In [ ]:
print("Cobertura da população:")

display(
    df_pop
    .groupBy("ano")
    .agg(
        F.countDistinct("id_municipio").alias("municipios"),
        F.sum("populacao").alias("populacao_total"),
    )
    .orderBy("ano")
)

print("Cobertura da RAIS:")

display(
    df_rais
    .groupBy("ano", "ano_rais")
    .agg(
        F.countDistinct("id_municipio").alias("municipios"),
        F.sum("quantidade_vinculos_ativos")
            .alias("vinculos_ativos"),
    )
    .orderBy("ano")
)


## 6. Integração socioeconômica

A população será utilizada como base territorial e a RAIS será integrada por:

`ano + id_municipio`


In [ ]:
df_socio = (
    df_pop.alias("pop")
    .join(
        df_rais.alias("rais"),
        on=["ano", "id_municipio"],
        how="left",
    )
    .select(
        F.col("ano"),
        F.col("id_municipio"),
        F.col("pop.sigla_uf").alias("sigla_uf"),
        F.col("populacao"),
        F.col("ano_rais"),
        F.col("quantidade_vinculos_ativos"),
        F.col("quantidade_vinculos_clt"),
        F.col("quantidade_vinculos_estatutarios"),
    )
)

display(
    df_socio
    .orderBy("ano", "id_municipio")
    .limit(20)
)


## 7. Feature engineering socioeconômico

Será criada a variável:

`vinculos_ativos_por_1000_habitantes`

Essa feature normaliza o número absoluto de vínculos formais pelo tamanho da população e facilita comparações entre municípios de diferentes portes.


In [ ]:
df_socio = (
    df_socio
    .withColumn(
        "vinculos_ativos_por_1000_habitantes",
        F.when(
            F.col("populacao") > 0,
            (
                F.col("quantidade_vinculos_ativos")
                / F.col("populacao")
            ) * F.lit(1000.0),
        )
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp(),
    )
)

display(
    df_socio
    .orderBy("ano", "id_municipio")
    .limit(20)
)


## 8. Validação da base integrada

As validações abaixo verificam:

- quantidade de municípios por ano;
- ausência de população nula;
- cobertura da RAIS;
- inexistência de duplicidade por `ano + id_municipio`.


In [ ]:
display(
    df_socio
    .groupBy("ano")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("id_municipio").alias("municipios"),
        F.sum(
            F.when(
                F.col("populacao").isNull(),
                1,
            ).otherwise(0)
        ).alias("sem_populacao"),
        F.sum(
            F.when(
                F.col("quantidade_vinculos_ativos").isNull(),
                1,
            ).otherwise(0)
        ).alias("sem_rais"),
    )
    .orderBy("ano")
)


In [ ]:
duplicados = (
    df_socio
    .groupBy("ano", "id_municipio")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Combinações duplicadas ano + id_municipio:",
    duplicados,
)

if duplicados != 0:
    raise ValueError(
        "Foram encontradas duplicidades na base socioeconômica."
    )


## 9. Cobertura sobre a base de alunos

Esta etapa mede quantos municípios presentes na `fato_alunos` possuem contexto socioeconômico disponível.


In [ ]:
df_alunos_municipios = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.fato_alunos"
    )
    .select(
        "ano",
        "id_municipio",
    )
    .distinct()
)

df_cobertura = (
    df_alunos_municipios.alias("a")
    .join(
        df_socio
        .select(
            "ano",
            "id_municipio",
            "populacao",
            "quantidade_vinculos_ativos",
        )
        .alias("s"),
        on=["ano", "id_municipio"],
        how="left",
    )
)

display(
    df_cobertura
    .groupBy("ano")
    .agg(
        F.count("*").alias("municipios_alunos"),
        F.sum(
            F.when(
                F.col("populacao").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("municipios_com_populacao"),
        F.sum(
            F.when(
                F.col("quantidade_vinculos_ativos").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("municipios_com_rais"),
    )
    .orderBy("ano")
)


## 10. Persistência na camada Silver

Após as validações, a base socioeconômica é persistida em Delta Lake.


In [ ]:
(
    df_socio
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print(
    f"Tabela criada com sucesso: {OUTPUT_TABLE}"
)

print(
    "Quantidade de registros:",
    spark.table(OUTPUT_TABLE).count(),
)


## 11. Validação final


In [ ]:
display(
    spark.sql(
        f"""
        SELECT
            ano,
            COUNT(*) AS registros,
            COUNT(DISTINCT id_municipio) AS municipios,
            ROUND(AVG(populacao), 2) AS populacao_media,
            SUM(quantidade_vinculos_ativos) AS vinculos_ativos,
            ROUND(
                AVG(vinculos_ativos_por_1000_habitantes),
                2
            ) AS media_vinculos_por_1000_habitantes
        FROM {OUTPUT_TABLE}
        GROUP BY ano
        ORDER BY ano
        """
    )
)


## Conclusão

Esta etapa constrói uma dimensão socioeconômica municipal consistente com os períodos utilizados na modelagem.

As principais features disponíveis são:

- `populacao`;
- `quantidade_vinculos_ativos`;
- `quantidade_vinculos_clt`;
- `quantidade_vinculos_estatutarios`;
- `vinculos_ativos_por_1000_habitantes`.

A RAIS foi utilizada com **defasagem de um ano**, reduzindo o risco de data leakage temporal.

O próximo passo será integrar essas variáveis à base individual de alunos e construir a Gold `base_modelagem_aluno`.
